# v3 — bigger dataset → QLoRA (4-bit, CUDA) → base-vs-tuned

The **canonical 4-bit QLoRA** run for the NAME-judgment core, on a Colab **GPU** runtime. Unlike the
Day-3 notebook (which regenerated a small v1 inline) and the v2 runs (local Apple-MPS *plain* LoRA, a
re-baseline), this notebook produces the real 4-bit QLoRA numbers on a **scaled v3 dataset**.

Pipeline:
1. Clone `main` (already carries the hardened, leak-free datagen pipeline + `merge.py` + co-occurrence).
2. **Generate the v3 dataset** from the frozen `configs/datagen.yaml` recipe at `scale=3.0`
   (matched minimal pairs + category singles + Faker negatives + CRAPII real slice), fold in the
   committed co-occurrence contrast examples, de-leak, split.
3. Train the **frozen** `configs/train.yaml` QLoRA (r=32/α=32, lr 2e-4, seq 2048, 3 epochs,
   completion-only) → `outputs/sft-v3`.
4. Score **base vs tuned-v3** on the 51 quarantined hard cases and print the delta.
5. Persist adapter + reports + splits to Google Drive (Colab VMs are ephemeral).

> **Day-4 rule (do not touch hyperparameters):** `configs/train.yaml` is frozen. v3 changes the
> **data only** — more minimal-pair contrast to cut over-tagging — not lr/r/epochs.

> ⚠️ **Teacher key required for generation.** Generation calls a frontier teacher (`gpt-4o` by
> default). As of this writing the project's OpenAI key returned `billing_not_active` — make sure
> billing is active, or set an `ANTHROPIC_API_KEY` instead. **No key?** Set `GENERATE = False` in
> §5 to train on the committed v2 splits (268 rows) as a fallback.

## 0. Setup — clone repo + install (GPU runtime)

Runtime → Change runtime type → **GPU** (T4 is fine; A100 is faster) before running.

In [ ]:
import os, sys

if not os.path.isdir("/content/slm-deid"):
    !git clone https://github.com/f15cubing/slm-deid.git /content/slm-deid
os.chdir("/content/slm-deid")
!git checkout main && git pull
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

# QLoRA stack (unsloth = 4-bit + bitsandbytes) + datagen/eval deps.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || pip install -q unsloth
!pip install -q trl datasets faker pyyaml kaggle openai anthropic

In [ ]:
# Confirm we actually have a CUDA GPU (unsloth 4-bit QLoRA needs one).
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime > Change runtime type > GPU."
print("CUDA device:", torch.cuda.get_device_name(0))

## 1. Credentials (teacher key + Kaggle)

Set your teacher key here (or via the Colab 🔑 Secrets panel). Set **one** of OpenAI / Anthropic —
the generator auto-picks Anthropic if its key is present, else OpenAI. Kaggle creds are only needed
if you want the CRAPII real slice (§3).

In [ ]:
import os

# --- teacher key: set ONE (leave the other unset) ---
# os.environ["OPENAI_API_KEY"] = "sk-..."      # gpt-4o teacher (default)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # claude teacher (alternative)

# --- Kaggle (for the CRAPII real slice; optional) ---
# os.environ["KAGGLE_USERNAME"] = "..."
# os.environ["KAGGLE_KEY"] = "..."

has_key = bool(os.environ.get("OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY"))
print("teacher key set:", has_key, "| provider:",
      "anthropic" if os.environ.get("ANTHROPIC_API_KEY") else ("openai" if os.environ.get("OPENAI_API_KEY") else "NONE"))

## 2. Mount Google Drive (persistence)

Colab VMs are ephemeral and `outputs/` is git-ignored — without this, the adapter + reports vanish
when the runtime recycles. Everything is copied to `MyDrive/slm-deid-v3/` at the end (§7).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/slm-deid-v3"
os.makedirs(DRIVE_OUT, exist_ok=True)
print("persisting to:", DRIVE_OUT)

## 3. (Optional) CRAPII real slice from Kaggle

The `configs/datagen.yaml` recipe folds in a small CRAPII real slice (`crapii_limit: 120`) alongside
the synthetic bulk. Skip this cell to generate a **synthetic-only** v3 (the generation cell auto-
detects whether the file is present and turns the CRAPII slice off if it isn't).

In [ ]:
import glob

crapii_path = None
if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
    if not glob.glob("data/raw/*.json*"):
        !mkdir -p data/raw
        !kaggle datasets download -d langdonholmes/cleaned-repository-of-annotated-pii -p data/raw --unzip
    hits = glob.glob("data/raw/*.json*")
    crapii_path = hits[0] if hits else None
print("CRAPII slice:", crapii_path or "(off — synthetic-only)")

## 4. Generate the v3 dataset (scaled)

Runs the frozen `configs/datagen.yaml` recipe (matched minimal pairs weighted to the categories v2
over-tagged, + category singles + Faker negatives + optional CRAPII), then folds in the committed
co-occurrence contrast examples, re-applies the token/surface/passage eval-leakage guards, and
re-splits.

- `SCALE` is the one size knob. `3.0` ≈ ~1,530 raw → ~800–1,000 kept after the quality + leakage
  gates (a genuine step up from v2's 268). Each raw example ≈ 2 teacher calls, so `scale=3.0` ≈
  ~3k sequential API calls (tens of minutes). Lower it (1.0–2.0) for a cheaper/faster first run.
- Set `GENERATE = False` to skip generation and train on the **committed v2 splits** (no key needed).

In [ ]:
GENERATE = True    # False -> use committed data/splits (v2, 268 rows) as-is
SCALE = 3.0        # size knob; ~1,530 raw @ 3.0. Lower for a quicker/cheaper run.

if GENERATE:
    assert has_key, "No teacher key — set one in §1, or set GENERATE=False to use the committed splits."
    from src.datagen.generate import _load_yaml, build_dataset, write_splits
    from src.datagen.teacher import TeacherGenerator

    cfg = _load_yaml("configs/datagen.yaml")
    cfg.scale = SCALE
    cfg.crapii_path = crapii_path  # None -> synthetic-only (build_dataset skips a missing path)

    if os.environ.get("ANTHROPIC_API_KEY"):
        from src.eval.judge import build_anthropic_complete as build_complete
    else:
        from src.eval.judge import build_openai_complete as build_complete
    complete = build_complete(temperature=0.7)  # >0 so scaled batches add NEW passages, not dups
    teacher = TeacherGenerator(gen=complete, verify=complete)

    train, val, drops = build_dataset(cfg, teacher)
    write_splits(train, val, "data")
    print(f"v3 synthetic+real: train={len(train)} val={len(val)} drops={drops}")
else:
    print("GENERATE=False — using committed v2 splits under data/splits/")

In [ ]:
# Fold in the committed co-occurrence contrast examples (same surface used as person + as a thing in
# ONE passage) and re-run the eval-leakage guards + a fresh split. Safe to run even when GENERATE
# is False (it just merges the co-occurrence set into the committed splits). Dedup drops repeats.
from src.datagen.merge import merge
from src.datagen.generate import write_splits

sources = ["data/splits/train.jsonl", "data/splits/val.jsonl"]
if os.path.exists("data/cooccur/cooccur.jsonl"):
    sources.append("data/cooccur/cooccur.jsonl")
train, val, stats = merge(sources, eval_dir="eval", val_frac=0.1, seed=0)
write_splits(train, val, "data")
print("v3 final splits:", stats)
assert stats["eval_leak"] == 0 and stats["eval_token_leak"] == 0 and stats["eval_surface_leak"] == 0, \
    "eval leakage detected — investigate before training"
!wc -l data/splits/train.jsonl data/splits/val.jsonl

## 5. QLoRA training (frozen config; 4-bit; 3 epochs) → `outputs/sft-v3`

`configs/train.yaml` is byte-identical to the v1/v2 runs — only the data changed. `output_dir` is
overridden to `outputs/sft-v3` so it sits beside the earlier adapters.

In [ ]:
from src.train.qlora import load_config, train

cfg = load_config("configs/train.yaml")
adapter = train(cfg, smoke=False, output_dir="outputs/sft-v3")
print("adapter:", adapter)

## 6. Base vs tuned-v3 on the quarantined hard cases

**Win condition (SPOV 7):** tuned beats base on the ambiguous categories — especially
**over_tag_rate ↓** and **f5 ↑** on `person_vs_eponym` / `person_vs_place` / `person_vs_common`, while
recall stays high. This is the first *4-bit* base-vs-tuned number (the v2 numbers were MPS plain-LoRA,
so they re-baseline against this run, not carry over).

In [ ]:
from src.eval.run import evaluate, compare, write_report, _load_examples
from src.infer import load_hf_tagger

examples = _load_examples("eval/hardcases")
print(f"{len(examples)} quarantined hard-case examples\n")

base = load_hf_tagger()                      # 4-bit Qwen3-1.7B base
rep_base = evaluate(base, examples, label="base")
write_report(rep_base)

tuned = load_hf_tagger(adapter=adapter)       # base + v3 LoRA adapter
rep_tuned = evaluate(tuned, examples, label="tuned-v3")
write_report(rep_tuned)

print(compare([rep_base, rep_tuned]))
print("\n--- per-category f5 / over_tag (base -> tuned-v3) ---")
for cat in sorted({e.category for e in examples}):
    b = rep_base.by_category.get(cat); t = rep_tuned.by_category.get(cat)
    if b and t:
        print(f"  {cat:18s} f5 {b.f5:.2f} -> {t.f5:.2f} | over_tag {b.over_tag_rate:.2f} -> {t.over_tag_rate:.2f}")

## 7. Persist to Drive

Copy the adapter, eval reports, and the exact v3 splits to Drive so nothing is lost when the VM
recycles. Bring the splits back down locally afterwards if you want to commit them as the v3 dataset.

In [ ]:
import shutil

for src_dir, name in [
    ("outputs/sft-v3", "sft-v3"),
    ("data/eval_reports", "eval_reports"),
    ("data/splits", "splits"),
]:
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f"{DRIVE_OUT}/{name}", dirs_exist_ok=True)
        print("saved", src_dir, "->", f"{DRIVE_OUT}/{name}")
print("\nDone. Paste the compare() table back and we'll record it in docs/results.md + plan v-next.")